# 03 — Huấn luyện & Đánh giá Mô hình
**Phụ trách:** Thạch  
**Yêu cầu:** Chạy `02_preprocessing.ipynb` trước

**Mục tiêu:** Train XGBoost / Random Forest / Logistic Regression, so sánh, chọn best model

In [ ]:
import sys; sys.path.append('..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
from src.models import train_model, evaluate_model, compare_models
sns.set_theme(style='whitegrid')
print('Setup OK')

## 1. Load processed data

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 2. Train models

Models được config trong `src/models.py`. Sau khi train tự lưu vào `models/`.

In [ ]:
# XGBoost — thường tốt nhất với tabular + imbalanced data
xgb = train_model('xgboost', X_train, y_train)

In [ ]:
rf = train_model('random_forest', X_train, y_train)

In [ ]:
# Logistic Regression — baseline để so sánh
lr = train_model('logistic_regression', X_train, y_train)

## 3. Đánh giá từng model

Metric chính: **PR-AUC** (Average Precision) — chính xác hơn ROC-AUC khi dữ liệu lệch

Trong fraud detection: ưu tiên **Recall cao** (bắt nhiều fraud) + Precision hợp lý (ít false alarm)

In [ ]:
r_xgb = evaluate_model(xgb, X_test, y_test, 'xgboost')

In [ ]:
r_rf  = evaluate_model(rf,  X_test, y_test, 'random_forest')

In [ ]:
r_lr  = evaluate_model(lr,  X_test, y_test, 'logistic_regression')

## 4. So sánh tổng hợp

In [ ]:
cmp = compare_models([r_xgb, r_rf, r_lr])

cmp[['roc_auc','pr_auc','f1_score']].plot(
    kind='bar', figsize=(10,5), colormap='Set2', alpha=0.85)
plt.title('So sánh hiệu năng các mô hình')
plt.ylabel('Score'); plt.ylim(0, 1.05); plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150)
plt.show()

## 5. Feature Importance (XGBoost)

In [ ]:
fi = pd.Series(xgb.feature_importances_,
               index=X_train.columns).sort_values(ascending=True)
fi.plot(kind='barh', figsize=(8,5), color='steelblue', alpha=0.8)
plt.title('Feature Importance — XGBoost')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150)
plt.show()
print('Top 3:', fi.tail(3).to_string())

## 6. Kết luận

In [ ]:
best = cmp['pr_auc'].idxmax()
print(f'Best model (PR-AUC): {best}')
print(cmp.loc[best].to_string())
print(f'Model saved: models/{best}.joblib')